# Digital Twin AI - Milestone 1: Setup, Infrastructure & Data Models

This notebook serves as a complete setup and verification guide for **Milestone 1** of the Digital Twin AI project in Google Colab.

### Milestone 1 Objectives:
1. Setup environment and install dependencies
2. Configure Firebase Admin credentials and environment variables
3. Initialize Firebase Admin SDK (Firestore & Auth)
4. Test Firestore read/write/delete connection
5. Test Firebase Authentication user creation and retrieval
6. Test CRUD operations across all core data models
7. Test Activity History audit logging
8. Test Dashboard metric aggregation queries
9. Run and test Flask backend locally
10. Clean up test user and document data

## Section 1: Environment Setup

Install the required Python packages for Flask, Firebase Admin SDK, CORS support, and environment variables.

In [ ]:
# Install required packages
!pip install flask firebase-admin flask-cors python-dotenv requests

## Section 2: Firebase Credentials Setup

Upload your `serviceAccountKey.json` downloaded from the Firebase Console:
1. Open [Firebase Console](https://console.firebase.google.com/)
2. Go to **Project Settings** > **Service accounts**
3. Click **Generate new private key**
4. Run the cell below to upload the JSON key file into Colab

In [ ]:
import os
from google.colab import files

print("Upload your Firebase serviceAccountKey.json")
uploaded = files.upload()

# Set environment variable
for filename in uploaded.keys():
    os.environ['FIREBASE_CREDENTIALS_PATH'] = filename
    print(f"Credentials loaded from: {filename}")

### Set Environment Variables
Set project IDs and client-side configuration variables.

In [ ]:
import os

# Flask and Firebase Server Config
os.environ['FIREBASE_PROJECT_ID'] = 'your-project-id'  # Replace with your project ID
os.environ['FLASK_SECRET_KEY'] = 'your-secret-key-change-this'
os.environ['FLASK_DEBUG'] = 'true'

# Firebase Client Config (for frontend)
os.environ['FIREBASE_API_KEY'] = 'your-api-key'
os.environ['FIREBASE_AUTH_DOMAIN'] = 'your-project.firebaseapp.com'
os.environ['FIREBASE_STORAGE_BUCKET'] = 'your-project.appspot.com'
os.environ['FIREBASE_MESSAGING_SENDER_ID'] = 'your-sender-id'
os.environ['FIREBASE_APP_ID'] = 'your-app-id'

print("Environment variables set successfully.")

## Section 3: Initialize Firebase Admin SDK

Initialize Firebase Admin with the uploaded credentials and connect to Firestore and Auth.

In [ ]:
import firebase_admin
from firebase_admin import credentials, firestore, auth

cred_path = os.environ.get('FIREBASE_CREDENTIALS_PATH', 'serviceAccountKey.json')
cred = credentials.Certificate(cred_path)

if not firebase_admin._apps:
    firebase_admin.initialize_app(cred)

db = firestore.client()
print("Firebase Admin SDK initialized successfully!")
print(f"Connected to project: {firebase_admin.get_app().project_id}")

## Section 4: Test Firestore Connection

Verify read, write, and delete operations on a test document in the `_test` collection.

In [ ]:
# Write test document
test_ref = db.collection('_test').document('connection_test')
test_ref.set({'status': 'connected', 'timestamp': firestore.SERVER_TIMESTAMP})

# Read it back
test_doc = test_ref.get()
print(f"Firestore connection test: {test_doc.to_dict()}")

# Clean up
test_ref.delete()
print("Test document cleaned up. Firestore is working!")

## Section 5: Test Authentication

Create and verify a test user via the Firebase Admin SDK.

In [ ]:
# Create a test user
try:
    test_user = auth.create_user(
        email='test@digitaltwin.ai',
        password='TestPassword123!',
        display_name='Test User'
    )
    print(f"Created test user: {test_user.uid}")
    print(f"Email: {test_user.email}")
    print(f"Display Name: {test_user.display_name}")
except auth.EmailAlreadyExistsError:
    test_user = auth.get_user_by_email('test@digitaltwin.ai')
    print(f"Test user already exists: {test_user.uid}")

In [ ]:
# Retrieve user info
user_info = auth.get_user(test_user.uid)
print(f"UID: {user_info.uid}")
print(f"Email: {user_info.email}")
print(f"Display Name: {user_info.display_name}")
print(f"Created: {user_info.user_metadata.creation_timestamp}")

## Section 6: Test CRUD Operations

Test creating, reading, updating, and querying documents across all core collections.

In [ ]:
uid = test_user.uid

# Create user profile
profile_data = {
    'uid': uid,
    'email': 'test@digitaltwin.ai',
    'display_name': 'Test User',
    'date_of_birth': '1995-06-15',
    'gender': 'male',
    'phone': '+1234567890',
    'location': 'New York, USA',
    'occupation': 'Software Engineer',
    'bio': 'Digital Twin AI test user',
    'created_at': firestore.SERVER_TIMESTAMP,
    'updated_at': firestore.SERVER_TIMESTAMP
}
db.collection('users').document(uid).set(profile_data)
print(f"Profile created for user: {uid}")

# Read profile
profile = db.collection('users').document(uid).get().to_dict()
print(f"Profile: {profile['display_name']} - {profile['email']}")

In [ ]:
# Create income record
income_data = {
    'uid': uid,
    'source': 'salary',
    'amount': 5000,
    'currency': 'USD',
    'frequency': 'monthly',
    'date': '2024-01-15',
    'notes': 'Monthly salary',
    'created_at': firestore.SERVER_TIMESTAMP,
    'updated_at': firestore.SERVER_TIMESTAMP
}
_, income_ref = db.collection('income').add(income_data)
print(f"Income record created: {income_ref.id}")

# Read income records
incomes = db.collection('income').where('uid', '==', uid).stream()
for doc in incomes:
    data = doc.to_dict()
    print(f"  {doc.id}: {data['source']} - ${data['amount']}")

# Update income
income_ref.update({'amount': 5500, 'updated_at': firestore.SERVER_TIMESTAMP})
print(f"Income updated to $5500")

# Verify update
updated = income_ref.get().to_dict()
print(f"Verified: ${updated['amount']}")

In [ ]:
# Test expense
expense_data = {
    'uid': uid,
    'category': 'food',
    'amount': 250,
    'currency': 'USD',
    'date': '2024-01-15',
    'description': 'Groceries',
    'is_recurring': True,
    'created_at': firestore.SERVER_TIMESTAMP,
    'updated_at': firestore.SERVER_TIMESTAMP
}
_, exp_ref = db.collection('expenses').add(expense_data)
print(f"Expense created: {exp_ref.id}")

# Test savings
savings_data = {
    'uid': uid,
    'type': 'bank',
    'amount': 10000,
    'currency': 'USD',
    'target_amount': 50000,
    'target_date': '2025-12-31',
    'notes': 'Emergency fund',
    'created_at': firestore.SERVER_TIMESTAMP,
    'updated_at': firestore.SERVER_TIMESTAMP
}
_, sav_ref = db.collection('savings').add(savings_data)
print(f"Savings created: {sav_ref.id}")

# Test study record
study_data = {
    'uid': uid,
    'subject': 'Machine Learning',
    'topic': 'Neural Networks',
    'duration_minutes': 120,
    'date': '2024-01-15',
    'notes': 'Deep learning fundamentals',
    'created_at': firestore.SERVER_TIMESTAMP,
    'updated_at': firestore.SERVER_TIMESTAMP
}
_, study_ref = db.collection('study_records').add(study_data)
print(f"Study record created: {study_ref.id}")

# Test academic record
academic_data = {
    'uid': uid,
    'course': 'CS229 Machine Learning',
    'institution': 'Stanford Online',
    'grade': 'A',
    'gpa': 3.9,
    'semester': 'Fall',
    'year': 2024,
    'status': 'in-progress',
    'created_at': firestore.SERVER_TIMESTAMP,
    'updated_at': firestore.SERVER_TIMESTAMP
}
_, acad_ref = db.collection('academic_records').add(academic_data)
print(f"Academic record created: {acad_ref.id}")

# Test habit
habit_data = {
    'uid': uid,
    'name': 'Morning Meditation',
    'category': 'health',
    'frequency': 'daily',
    'target_count': 1,
    'current_streak': 15,
    'is_active': True,
    'created_at': firestore.SERVER_TIMESTAMP,
    'updated_at': firestore.SERVER_TIMESTAMP
}
_, habit_ref = db.collection('habits').add(habit_data)
print(f"Habit created: {habit_ref.id}")

# Test fitness record
fitness_data = {
    'uid': uid,
    'activity_type': 'running',
    'duration_minutes': 45,
    'calories_burned': 350,
    'distance_km': 5.2,
    'date': '2024-01-15',
    'notes': 'Morning run',
    'created_at': firestore.SERVER_TIMESTAMP,
    'updated_at': firestore.SERVER_TIMESTAMP
}
_, fit_ref = db.collection('fitness_records').add(fitness_data)
print(f"Fitness record created: {fit_ref.id}")

# Test goal
goal_data = {
    'uid': uid,
    'title': 'Complete ML Course',
    'description': 'Finish Stanford CS229',
    'category': 'academic',
    'target_date': '2024-06-30',
    'status': 'in-progress',
    'priority': 'high',
    'progress_percent': 35,
    'created_at': firestore.SERVER_TIMESTAMP,
    'updated_at': firestore.SERVER_TIMESTAMP
}
_, goal_ref = db.collection('goals').add(goal_data)
print(f"Goal created: {goal_ref.id}")

print("\nAll CRUD tests passed!")

## Section 7: Test Activity History

Log a user activity and retrieve the recent history ordered chronologically.

In [ ]:
# Log a test activity
activity_data = {
    'uid': uid,
    'action': 'create',
    'entity_type': 'income',
    'entity_id': income_ref.id,
    'summary': 'Created income: salary',
    'details': {'source': 'salary', 'amount': 5000},
    'timestamp': firestore.SERVER_TIMESTAMP
}
db.collection('activity_history').add(activity_data)

# Read activity history
activities = db.collection('activity_history').where('uid', '==', uid).order_by('timestamp', direction=firestore.Query.DESCENDING).limit(10).stream()
print("Activity History:")
for doc in activities:
    data = doc.to_dict()
    print(f"  [{data['action'].upper()}] {data['entity_type']}: {data['summary']}")

## Section 8: Test Dashboard Aggregation

Calculate aggregated summary metrics across income, expenses, savings, study records, and active goals.

In [ ]:
# Total income
incomes = db.collection('income').where('uid', '==', uid).stream()
total_income = sum(doc.to_dict()['amount'] for doc in incomes)

# Total expenses
expenses = db.collection('expenses').where('uid', '==', uid).stream()
total_expenses = sum(doc.to_dict()['amount'] for doc in expenses)

# Total savings
savings = db.collection('savings').where('uid', '==', uid).stream()
total_savings = sum(doc.to_dict()['amount'] for doc in savings)

# Study hours
studies = db.collection('study_records').where('uid', '==', uid).stream()
total_study_minutes = sum(doc.to_dict()['duration_minutes'] for doc in studies)

# Active goals
goals = db.collection('goals').where('uid', '==', uid).stream()
active_goals = sum(1 for doc in goals if doc.to_dict()['status'] in ('not-started', 'in-progress'))

print(f"Dashboard Summary:")
print(f"  Total Income: ${total_income}")
print(f"  Total Expenses: ${total_expenses}")
print(f"  Net: ${total_income - total_expenses}")
print(f"  Total Savings: ${total_savings}")
print(f"  Study Hours: {total_study_minutes / 60:.1f}")
print(f"  Active Goals: {active_goals}")

## Section 9: Run Flask Backend (Optional)

Run the Flask backend server in a background thread within Colab to test REST API endpoints locally.

In [ ]:
# Option 1: Run Flask in a thread (for API testing)
import threading
from app import create_app

app = create_app()

def run_flask():
    app.run(host='0.0.0.0', port=5000, use_reloader=False)

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()
print("Flask server started on port 5000")

In [ ]:
import requests

response = requests.get('http://localhost:5000/health')
print(f"Health check: {response.json()}")

## Section 10: Cleanup

Delete all test records created in Firestore collections and delete the test Firebase Auth user.

In [ ]:
# Delete test data from Firestore
for collection_name in ['income', 'expenses', 'savings', 'study_records', 'academic_records', 'habits', 'fitness_records', 'goals', 'activity_history']:
    docs = db.collection(collection_name).where('uid', '==', uid).stream()
    for doc in docs:
        doc.reference.delete()
    print(f"Cleaned {collection_name}")

# Delete user profile
db.collection('users').document(uid).delete()
print("Cleaned user profile")

# Delete Firebase Auth user
auth.delete_user(uid)
print(f"Deleted test user: {uid}")

print("\nAll test data cleaned up!")